# Does the leading run of identical letters set kappa? A fresh-rule and mutation test

## Question and why it matters
`kappa` is the pattern-field coefficient of the t^(1/3) growth law. A post hoc finding (`../2026-10-16-kappa-prefix-decode`) said the number of identical opening letters of a rule (`lead`) predicts ln kappa beyond early area ln A(22). Here it is tested on 300 fresh, lead-stratified rules (lengths 13/14) and by single-letter mutants. See `README.md`, `NOTEBOOK.md`, `notes/open-questions-hypotheses.md`.

## Assumptions and definitions
- `lead`: length of the opening run of one letter, capped at 6 in models. Stationary: |g_k| < 0.05; kept: >= 6 core-band runs (alpha_w in [0.25, 0.42], kappa > 0) of 8 seeds. *Assumed, unverified*: dropout from the kept set does not bias the slope (README Part A bounds it only by reweighting by lead).
- `ln A22`: log area at probe window k=22, a post-treatment quantity in the mutant experiment (it changes when a letter is flipped), so adjusting for it is not a clean causal estimate.
- Predictions L1-L6, A1-A2, M1-M4 were registered in the README before the measurement; those that failed are reported as failed.

## Provenance

In [1]:
import hashlib, json, subprocess, datetime
from pathlib import Path
import numpy as np, polars as pl
SEED = 20261016; np.random.seed(SEED)
root = Path.cwd()
while not (root / "AGENTS.md").exists():
    assert root != root.parent, "repo root not found"; root = root.parent
E = root / "experiments/2026-10-16-kappa-leadrun-fresh/results"
files = ["analysis.json", "analysis.txt", "followup.json", "followup.txt", "fresh_rules.csv", "mutants_analysis.json", "mutants_analysis.txt", "mutant_pairs.csv", "mutants.csv", "rules.txt"]
for f in files:
    assert (E / f).exists(), f
    print(f"{f:24s} {hashlib.sha256((E / f).read_bytes()).hexdigest()[:16]}")
print("git", subprocess.run(["git", "rev-parse", "--short", "HEAD"], cwd=root, capture_output=True, text=True).stdout.strip(), "seed", SEED, datetime.date.today())

analysis.json            a2977ebf303393ba
analysis.txt             df3eb3d194b6c98e
followup.json            502e4f502b628329
followup.txt             de75bcf3ffc0d175
fresh_rules.csv          7fd5b7a8aa6d8312
mutants_analysis.json    ab82c60c0c88ff28
mutants_analysis.txt     88ddf3c872de873f
mutant_pairs.csv         213bf784fe9bda43
mutants.csv              f25ed85e9cab71e9
rules.txt                01703ee3bda8033d
git 5b97961 seed 20261016 2026-10-16


## Data and method
Heavy computation lives in `scripts/` (`sample.py`, `analyse.py`, `followup.py`, `mutants.py`, `analyse_mutants.py`) and the Rust `probe 27 8192` binary; this notebook only loads committed outputs. The census training set (2183 stationary rules) is from `2026-10-kappa-rule-space`.

## Results 1: the lead effect on fresh rules

In [2]:
print((E / "analysis.txt").read_text())

# cmd: uv run experiments/2026-10-16-kappa-leadrun-fresh/scripts/analyse.py  git: 58b4f38 (tree dirty)  seed 20261016
runs 2400, escaped 365, fitted 2006, core 1488; rules with >=6 core runs 183/300; stationary 176
L5 retention by lead stratum (of 50 sampled): lead, kept(>=6 core runs), stationary
  1: 0.74 0.68  (n=50)
  2: 0.64 0.64  (n=50)
  3: 0.70 0.68  (n=50)
  4: 0.62 0.62  (n=50)
  5: 0.56 0.52  (n=50)
  6: 0.40 0.38  (n=50)
L5 |lead1-lead6| both < 0.20: FAILED
census stationary train n=2183; fresh stationary test n=176; spline-only strict R2 on fresh = 0.124
fresh stationary rules per lead: 1:34, 2:32, 3:34, 4:31, 5:26, 6:19
corr(lead, ln A22) fresh -0.721; census -0.568
L1 gain of lead term +0.651 CI [+0.498,+0.828]; census slope b = +0.195/step -> HELD
L2 census slope +0.195 in [0.07,0.13]: False; fresh refit slope +0.152 CI [+0.129,+0.174] -> FAILED
L3 tertile slopes [0.151, 0.155, 0.149]; pooled within-tertile +0.152 CI [+0.128,+0.177] -> HELD
L4 shuffled-lead gain median 

**Shows:** the held-out gain of the lead term over a ln A22 spline is large and its slope is positive within area tertiles (L1, L3 held). **Does not show:** that the census slope (+0.195) transfers unchanged (L2 failed: fresh +0.152), nor that survivors represent all rules (L5 failed: retention falls with lead). L4 failed as a mis-specified control; see the next table.

## Results 2: nulls and selection bounds

In [3]:
print((E / "followup.txt").read_text())
t = pl.read_csv(E / "fresh_rules.csv")
print(t.group_by("lead").agg(rules=pl.len(), kept=pl.col("kept").sum(), stationary=pl.col("stationary").sum(), mean_lnk=pl.col("lnk").mean()).sort("lead"))

# cmd: uv run experiments/2026-10-16-kappa-leadrun-fresh/scripts/followup.py  git: 78bcafd (tree dirty)  seed 20261016
fresh stationary n=176, census stationary n=2183; observed gain +0.651
A1 unrestricted         gain median -0.001 p95 +0.025 p99 +0.038 max +0.049  (obs +0.651; |median|<=0.02: True)
A1 within ln A quintile gain median +0.000 p95 +0.033 p99 +0.051 max +0.058  (obs +0.651; |median|<=0.02: True)
A2 slope [all leads                       ] +0.152 CI [+0.130,+0.174] n=176
A2 slope [leads 1-3                       ] +0.235 CI [+0.196,+0.271] n=100
A2 slope [leads 1-4                       ] +0.197 CI [+0.168,+0.225] n=131
A2 slope [length 13                       ] +0.144 CI [+0.110,+0.183] n=93
A2 slope [length 14                       ] +0.157 CI [+0.132,+0.182] n=83
A2 slope [all, inverse-retention weights  ] +0.144 CI [+0.123,+0.167] n=176
A2 dropouts by lead (50 sampled each): rules with >=1 escaped run, mean escaped runs/rule, kept, stationary
  lead 1: any_escape 3/5

**Shows:** a coefficient refit on a shuffled label gives gain ~0 (so +0.65 is not a fitting artefact) and the slope is positive in every cut. **Does not show:** that the slope is linear: it is steeper at leads 1-3 (+0.235) than overall, and high-lead strata are selection-exposed.

## Results 3: single-letter mutants (257 pairs)

In [4]:
print((E / "mutants_analysis.txt").read_text())
p = pl.read_csv(E / "mutant_pairs.csv")
print(p.group_by("arm").agg(n=pl.len(), mean_d=pl.col("d").mean(), mean_dA=pl.col("dA").mean(), mean_dlead=pl.col("dlead").mean()).sort("arm"))

# cmd: uv run experiments/2026-10-16-kappa-leadrun-fresh/scripts/analyse_mutants.py  git: 5b97961 (tree dirty)  seed 20261016
mutants 417, with base stationary-fresh row 417, mutant kept 269, kept+stationary pairs 257
arm E: n=81 mean d=-0.062 CI [-0.152,+0.025]  mean dlnA22=-0.292 [-0.380,-0.205]  mean dlead=+1.64
arm S: n=73 mean d=-0.198 CI [-0.290,-0.110]  mean dlnA22=+0.013 [-0.068,+0.089]  mean dlead=-1.00
arm C: n=103 mean d=-0.006 CI [-0.049,+0.037]  mean dlnA22=-0.000 [-0.037,+0.036]  mean dlead=+0.00
M1 arm E mean d -0.062 CI lower -0.152; pooled d ~ c0 + b*dlead + c*dA: b=+0.158 [+0.139,+0.179], c=+1.037 [+0.957,+1.119], c0=-0.023 -> FAILED
M2 E - C mean d -0.056 CI [-0.159,+0.040] -> FAILED
M3 arm S mean d -0.198 CI upper -0.110 -> HELD
M4 retention of mutant given base stationary-fresh: arm, mutants, kept, kept+stationary
  E: 152, 82, 81
  S: 113, 82, 73
  C: 152, 105, 103
desc arm E: d/dlead mean -0.016; |dA|<0.15 pairs n=26, mean d +0.193
desc arm S: d/dlead mean +0.198

**Shows:** shortening the lead lowers ln kappa (-0.20) at unchanged area, control flips do nothing, and the pooled regression slope on dlead (+0.158) reproduces the cross-sectional slope. **Does not show:** a net raw effect of extending the lead (-0.06, CI includes 0, because area drops 0.29 and ln kappa tracks ln A22 with coefficient ~1), so registered M1 and M2 failed. It also does not separate "run length" from "the letter at index k".

## Caveats and controls
- Negative controls: refit-on-shuffle (A1) and the late-letter control arm C (d = -0.006).
- Failed predictions: L2 (range), L4 (control mis-specified), L5, the A2 "within 0.05" clause, M1, M2. None repaired.
- ln A22 is post-treatment; adjusted and area-matched mutant estimates are not unbiased causal effects. Sample sizes: 176 fresh stationary rules, 257 mutant pairs; one seed set (601-608) throughout.

## What this changes / next steps
The lead association is real and survives fresh rules, shuffling nulls and within-rule pairing, but it is entangled with early area. Next: flips that leave early area unchanged, kappa from late windows only, and larger mutant sweeps (a GPU port of the probe statistics is in progress separately).

## Open questions
Is ln kappa nearly redundant with ln A22 (coefficient ~1)? Does the letter at index k, rather than run length, carry the effect? Why is the dependence concave in lead?